# CertVIC immutable model snapshot provisioning
Provisioning-only: internet must be ON and accelerator OFF. Set the single `PROVIDER` parameter to one of the three locked values.

In [ ]:
PROVIDER = 'qwen2_5_vl_7b'  # exact alternatives: internvl_8b, llava_onevision_7b


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
SPECS = {
 'qwen2_5_vl_7b': {'repo':'Qwen/Qwen2.5-VL-7B-Instruct','commit':'cc594898137f460bfe9f0759e9844b3ce807cfb5','out':'qwen2_5_vl_7b_snapshot.zip'},
 'internvl_8b': {'repo':'OpenGVLab/InternVL2-8B','commit':'6fb9ad6924f69424e57fab2ab061d707688f0296','out':'internvl2_8b_snapshot.zip'},
 'llava_onevision_7b': {'repo':'llava-hf/llava-onevision-qwen2-7b-ov-hf','commit':'0d50680527681998e456c7b78950205bedd8a068','out':'llava_onevision_7b_snapshot.zip'}
}
if PROVIDER not in SPECS: raise ValueError(f'unsupported PROVIDER: {PROVIDER}')
CODE = Path('/kaggle/input/certvic-code-bundle/certvic_code_bundle.zip')
WORK = Path('/kaggle/working/certvic_snapshot_builder')
if not CODE.is_file(): raise FileNotFoundError(CODE)
with zipfile.ZipFile(CODE) as z: z.extractall(WORK)
os.chdir(WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface_hub==0.26.2'], check=True)
from huggingface_hub import snapshot_download
spec = SPECS[PROVIDER]
SNAPSHOT = Path('/kaggle/working/snapshot') / PROVIDER
snapshot_download(repo_id=spec['repo'], revision=spec['commit'], local_dir=SNAPSHOT)
symlinks = [str(p) for p in SNAPSHOT.rglob('*') if p.is_symlink()]
if symlinks: raise RuntimeError(f'snapshot contains symlinks: {symlinks[:5]}')
OUT1 = Path('/kaggle/working') / spec['out']
OUT2 = OUT1.with_name(OUT1.stem + '.rebuild.zip')
cmd = [sys.executable, '-m', 'certvic.cvpr.snapshot_bundle_builder', '--provider', PROVIDER, '--snapshot-root', str(SNAPSHOT), '--model-commit', spec['commit'], '--processor-commit', spec['commit'], '--structural-smoke-only']
subprocess.run(cmd + ['--output', str(OUT1)], check=True)
subprocess.run(cmd + ['--output', str(OUT2)], check=True)
def sha(p):
    digest = hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()
if sha(OUT1) != sha(OUT2): raise RuntimeError('snapshot rebuild is not byte-identical')
OUT2.unlink()
print({'status': 'IMMUTABLE_SNAPSHOT_BUILT_DETERMINISTIC', 'provider': PROVIDER, 'commit': spec['commit'], 'path': str(OUT1), 'sha256': sha(OUT1), 'size': OUT1.stat().st_size, 'paper_evidence': False})
